# Anthropic Responsible Scaling Policy v3.0: Interactive Visual Explorer

> RSP v3.0 went into effect February 24, 2026, replacing the 2023 policy. Two-tier mitigation: what Anthropic will do unilaterally vs what is framed as an industry-wide recommendation (including RAND SL-4 security standards). Adds Frontier Safety Roadmaps and Risk Reports as standing documents rather than one-off deliverables. Drops the 2023 pause commitment. Introduces the AI R&D-4 threshold: once crossed, Anthropic must publish an affirmative case identifying misalignment risks and mitigations. Claude Opus 4.6 does not cross it. Anthropic states in the v3.0 announcement that "confidently ruling this out is becoming difficult." SaferAI rated the 2023 RSP at 2.2; they downgraded v3.0 to 1.9, putting Anthropic in the "weak" RSP category alongside OpenAI and DeepMind. Qualitative thresholds replaced the 2023 quantitative commitments; removing the pause clause is the sharpest regression.

Welcome to the interactive companion notebook for **Anthropic Responsible Scaling Policy v3.0**.

In this notebook, you can interactively execute the lesson's raw implementation, plot state transformations, and run experiment variations.


In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt

# Configure plotting aesthetics
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (8, 5)
plt.rcParams['font.size'] = 11


In [ ]:
"""RSP v3.0 threshold evaluator — stdlib Python.

Mirrors the decision shape of Anthropic's RSP v3.0 for the AI R&D-4
threshold. Given a candidate model's capability measurements, decide
whether the threshold is crossed and what the affirmative case must
cover.

This is pedagogical: the real RSP involves human judgment across a
larger evidence base. The code is a reading aid, not a policy tool.
"""


In [ ]:
from __future__ import annotations

from dataclasses import dataclass

@dataclass
class CapabilityMeasurement:
    model_name: str
    # Fraction of internal AI R&D tasks the model can complete at
    # expert-human cost-equivalent (0.0-1.0).
    rd_automation_share: float
    # METR 50% time horizon in hours.
    metr_horizon_hours: float
    # Fraction of alignment-research pilot tasks the model completes
    # at or above human baseline (Anthropic AAR benchmark).
    aar_outperform_share: float
    # Evaluation-context gaming rate (0-1; 0 = never distinguishes).
    eval_context_gaming_rate: float


In [ ]:
# Thresholds per RSP v3.0 framing. Numbers are illustrative.
AI_RD_4_THRESHOLDS = {
    "rd_automation_share": 0.5,
    "metr_horizon_hours": 40.0,
    "aar_outperform_share": 0.4,
}

def threshold_crossed(m: CapabilityMeasurement) -> tuple[bool, list[str]]:
    reasons = []
    if m.rd_automation_share >= AI_RD_4_THRESHOLDS["rd_automation_share"]:
        reasons.append(
            f"rd_automation_share={m.rd_automation_share:.2f} "
            f">= {AI_RD_4_THRESHOLDS['rd_automation_share']}"
        )
    if m.metr_horizon_hours >= AI_RD_4_THRESHOLDS["metr_horizon_hours"]:
        reasons.append(
            f"metr_horizon_hours={m.metr_horizon_hours:.1f} "
            f">= {AI_RD_4_THRESHOLDS['metr_horizon_hours']}"
        )
    if m.aar_outperform_share >= AI_RD_4_THRESHOLDS["aar_outperform_share"]:
        reasons.append(
            f"aar_outperform_share={m.aar_outperform_share:.2f} "
            f">= {AI_RD_4_THRESHOLDS['aar_outperform_share']}"
        )
    crossed = len(reasons) >= 2  # any two triggers; illustrative
    return crossed, reasons


In [ ]:
def affirmative_case_template(m: CapabilityMeasurement) -> list[str]:
    sections = [
        "1. Capability inventory: specific measurements against RSP thresholds",
        "2. Misalignment risk analysis: modes the model could exhibit",
        "3. Evaluation-context gap: residual risk from eval-vs-deploy divergence",
        "4. Mitigation design: technical + operational + deployment gates",
        "5. Residual risk acknowledgement: what we cannot rule out",
        "6. Review: internal Safety Advisory Group sign-off + external reviewer",
    ]
    if m.eval_context_gaming_rate > 0.2:
        sections.append(
            f"7. Gaming-adjusted capability estimate "
            f"(observed gaming rate {m.eval_context_gaming_rate:.0%})"
        )
    return sections


In [ ]:
def evaluate(m: CapabilityMeasurement) -> None:
    crossed, reasons = threshold_crossed(m)
    print(f"\nModel: {m.model_name}")
    print("-" * 70)
    print(f"  rd_automation_share={m.rd_automation_share:.2f}  "
          f"metr_horizon_hours={m.metr_horizon_hours:.1f}  "
          f"aar_outperform_share={m.aar_outperform_share:.2f}  "
          f"gaming_rate={m.eval_context_gaming_rate:.0%}")
    if crossed:
        print("  AI R&D-4 threshold: CROSSED")
        for r in reasons:
            print(f"    - {r}")
        print("  required: affirmative case covering:")
        for section in affirmative_case_template(m):
            print(f"    {section}")
    else:
        print("  AI R&D-4 threshold: not crossed")
        if reasons:
            print("  single trigger(s) observed (below threshold):")
            for r in reasons:
                print(f"    - {r}")


In [ ]:
def main() -> None:
    print("=" * 70)
    print("RSP v3.0 AI R&D-4 THRESHOLD EVALUATOR (Phase 15, Lesson 19)")
    print("=" * 70)

# Claude Opus 4.6 per the v3.0 announcement: does not cross.
    opus_4_6 = CapabilityMeasurement(
        model_name="Claude Opus 4.6 (as stated by Anthropic in v3.0)",
        rd_automation_share=0.30,
        metr_horizon_hours=14.0,
        aar_outperform_share=0.35,
        eval_context_gaming_rate=0.12,
    )
    evaluate(opus_4_6)


In [ ]:
# Synthetic near-threshold model: Anthropic's concern is this class.
    near = CapabilityMeasurement(
        model_name="Synthetic next-gen (illustrative only)",
        rd_automation_share=0.55,
        metr_horizon_hours=48.0,
        aar_outperform_share=0.45,
        eval_context_gaming_rate=0.28,
    )
    evaluate(near)

print()
    print("=" * 70)
    print("HEADLINE: reading the policy is a practical skill")
    print("-" * 70)
    print("  Thresholds are qualitative in v3.0, not quantitative as in v2.")
    print("  The pause commitment from 2023 is removed; the affirmative case")
    print("  shape replaces it.")
    print("  SaferAI downgraded v3.0 from 2.2 to 1.9 (weak RSP category).")
    print("  Eval-context gaming (Lesson 1) biases capability numbers upward")
    print("  from the deploy-context reality; v3.0 acknowledges this.")


In [ ]:
if __name__ == "__main__":
    main()
